In [2]:
import numpy as np
import pandas as pd

# ── Datos de rentabilidad mensual (%) extraídos de la imagen ──────────────────
data = {
    "Año":  [2022, 2022, 2022, 2022, 2022, 2022, 2022, 2022, 2022,
             2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023,
             2024, 2024, 2024, 2024, 2024, 2024, 2024, 2024, 2024, 2024, 2024, 2024,
             2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025,
             2026, 2026, 2026, 2026, 2026, 2026],
    "Mes":  ["Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
             "Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
             "Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
             "Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
             "Ene", "Feb", "Mar", "Abr", "May", "Jun"],
    "Rentabilidad": [-2.78, 1.72, -9.17, 8.64, -1.40, 4.72, 7.37, 1.83, -8.13,
                     19.22, -7.16, -9.97, -5.17, -2.13, 6.88, 3.87, 4.85, -1.87, -4.14, 8.00, 2.64,
                     7.33, 1.28, 3.86, 0.00, 3.04, -12.66, 4.69, 0.37, 3.47, -2.49, -0.21, -1.63,
                     7.27, 4.52, 0.12, 3.60, 6.41, 3.97, 4.60, 7.00, 5.35, 1.10, 1.22, 5.35,
                     6.01, 5.08, -10.07, 6.27, 1.87, 2.18],
}

df = pd.DataFrame(data)
r = df["Rentabilidad"].values  # rentabilidades en %

# ── 1. Volatilidad mensual (desviación estándar de la muestra) ────────────────
vol_mensual = np.std(r, ddof=1)

# ── 2. Volatilidad anualizada (×√12) ─────────────────────────────────────────
vol_anual = vol_mensual * np.sqrt(12)

# ── 3. Volatilidad anualizada por año ────────────────────────────────────────
vol_por_ano = (
    df.groupby("Año")["Rentabilidad"]
    .std(ddof=1)                         # desv. estándar mensual por año
    .rename("Vol_mensual_%")
    .to_frame()
)
vol_por_ano["Vol_anualizada_%"] = vol_por_ano["Vol_mensual_%"] * np.sqrt(12)

# ── 4. Rentabilidad media y ratio de Sharpe simplificado (rf = 0) ─────────────
media_mensual   = np.mean(r)
media_anual     = media_mensual * 12          # aproximación lineal
sharpe_aprox    = media_anual / vol_anual

# ── Resultados ────────────────────────────────────────────────────────────────
print("=" * 52)
print("  ANÁLISIS DE VOLATILIDAD DEL PORTFOLIO")
print("=" * 52)
print(f"\n  Nº de meses analizados : {len(r)}")
print(f"  Rentabilidad media mens.: {media_mensual:>7.2f} %")
print(f"  Rentabilidad media anual: {media_anual:>7.2f} %  (×12)")
print(f"\n  Volatilidad mensual     : {vol_mensual:>7.2f} %")
print(f"  Volatilidad anualizada  : {vol_anual:>7.2f} %  (×√12)")
print(f"\n  Sharpe aprox. (rf=0)    : {sharpe_aprox:>7.3f}")

print("\n  Volatilidad por año:")
print(f"  {'Año':<6} {'Vol. mens.':>12} {'Vol. anual.':>13}")
print("  " + "-" * 33)
for año, row in vol_por_ano.iterrows():
    print(f"  {año:<6} {row['Vol_mensual_%']:>11.2f}% {row['Vol_anualizada_%']:>12.2f}%")

print("\n  Nota: volatilidad calculada como desviación")
print("  estándar muestral (ddof=1) de retornos simples.")
print("=" * 52)


inicio_filtro = df[(df["Año"] == 2024) & (df["Mes"] == "Jun")].index[0]
df_2y = df.loc[inicio_filtro:].copy()

# Convertimos los retornos a tanto por uno para cálculos matemáticos precisos
returns_2y = df_2y["Rentabilidad"] / 100

# 3. Cálculos estadísticos y cuantitativos
mean_monthly = returns_2y.mean()
vol_monthly = returns_2y.std(ddof=1)  # Desviación estándar muestral
vol_annualized = vol_monthly * np.sqrt(12)

# Semi-volatilidad a la baja (Downside Deviation respecto a un target de 0%)
target_return = 0.0
downside_returns = returns_2y[returns_2y < target_return]
# Al estilo institucional (Sortino ratio standard), se divide por el N total de la muestra filtrada
downside_deviation_monthly = np.sqrt(
    np.sum((downside_returns - target_return) ** 2) / len(returns_2y)
)
downside_deviation_annualized = downside_deviation_monthly * np.sqrt(12)

worst_month = returns_2y.min()

# 4. Consola de resultados estructurada
print("=" * 50)
print(f"ANÁLISIS DE VOLATILIDAD (Jun 2024 - Jun 2026) | N = {len(returns_2y)}")
print("=" * 50)
print(f"Rentabilidad Media Mensual:   {mean_monthly*100:6.2f}%")
print(f"Volatilidad Mensual (σ):      {vol_monthly*100:6.2f}%")
print(f"Volatilidad Anualizada:       {vol_annualized*100:6.2f}%")
print("-" * 50)
print(f"MÉTRICAS DE RIESGO DE COLA / BAJADA:")
print(
    f"Semi-Volatilidad Anualizada:  {downside_deviation_annualized*100:6.2f}%"
)
print(f"Peor Retorno Mensual (Max DD): {worst_month*100:6.2f}%")
print("=" * 50)

  ANÁLISIS DE VOLATILIDAD DEL PORTFOLIO

  Nº de meses analizados : 51
  Rentabilidad media mens.:    1.70 %
  Rentabilidad media anual:   20.40 %  (×12)

  Volatilidad mensual     :    5.77 %
  Volatilidad anualizada  :   19.99 %  (×√12)

  Sharpe aprox. (rf=0)    :   1.021

  Volatilidad por año:
  Año      Vol. mens.   Vol. anual.
  ---------------------------------
  2022          6.29%        21.79%
  2023          8.03%        27.83%
  2024          5.03%        17.44%
  2025          2.35%         8.13%
  2026          6.16%        21.32%

  Nota: volatilidad calculada como desviación
  estándar muestral (ddof=1) de retornos simples.
ANÁLISIS DE VOLATILIDAD (Jun 2024 - Jun 2026) | N = 25
Rentabilidad Media Mensual:     2.14%
Volatilidad Mensual (σ):        4.89%
Volatilidad Anualizada:        16.93%
--------------------------------------------------
MÉTRICAS DE RIESGO DE COLA / BAJADA:
Semi-Volatilidad Anualizada:   11.40%
Peor Retorno Mensual (Max DD): -12.66%
